In [7]:
import sys, os
os.chdir('..')
sys.path.append('.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_ratings, load_movies, get_train_test_split
from src.baseline_model import PopularityRecommender
from src.collaborative_filtering import CollaborativeFilteringModel

%matplotlib inline

ratings = load_ratings()
movies = load_movies()
train, test = get_train_test_split(ratings)

cf_model = CollaborativeFilteringModel(n_factors=20, n_epochs=20)
cf_model.fit(train)

print("Model trained successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/ml-100k/u.data'

In [ ]:
# Test set ke har rating ke liye, model ka prediction nikalte hain
test_predictions = test.copy()
test_predictions['predicted_rating'] = test_predictions.apply(
    lambda row: cf_model.predict(row['user_id'], row['item_id']), axis=1
)
test_predictions['error'] = abs(test_predictions['rating'] - test_predictions['predicted_rating'])

print(f"Mean Absolute Error (MAE): {test_predictions['error'].mean():.4f}")
print(f"Max Error: {test_predictions['error'].max():.4f}")
print(f"Min Error: {test_predictions['error'].min():.4f}")

test_predictions[['user_id', 'item_id', 'rating', 'predicted_rating', 'error']].head(10)

Mean Absolute Error (MAE): 0.7347
Max Error: 4.0000
Min Error: 0.0000


,user_id,item_id,rating,predicted_rating,error
75721,877,381,4,3.737487,0.262513
80184,815,602,3,3.808479,0.808479
19864,94,431,4,3.447283,0.552717
76699,416,875,2,3.099360,1.099360
92991,500,182,2,4.326366,2.326366
76434,259,1074,3,2.830514,0.169486
84004,598,286,5,3.749633,1.250367
80917,886,496,4,4.168531,0.168531
60767,837,15,3,3.033390,0.033390
50074,521,184,4,3.708021,0.291979


In [ ]:
# Top 10 worst predictions (jahan error sabse zyada hai)
worst_predictions = test_predictions.sort_values('error', ascending=False).head(10)

worst_with_titles = worst_predictions.merge(
    movies[['item_id', 'title']], on='item_id'
)[['user_id', 'title', 'rating', 'predicted_rating', 'error']]

print("Top 10 Worst Predictions:")
worst_with_titles

Top 10 Worst Predictions:


,user_id,title,rating,predicted_rating,error
0,1,Babe (1995),1,5.000000,4.000000
1,405,Another Stakeout (1993),5,1.000000,4.000000
2,312,Die Hard (1988),1,4.983202,3.983202
3,38,Ben-Hur (1959),1,4.854037,3.854037
4,495,"Sound of Music, The (1965)",1,4.816545,3.816545
5,825,Scream (1996),1,4.797770,3.797770
6,887,Little Women (1994),1,4.677330,3.677330
7,239,Annie Hall (1977),1,4.663370,3.663370
8,137,Legends of the Fall (1994),1,4.630611,3.630611
9,239,"Shawshank Redemption, The (1994)",1,4.587899,3.587899


In [ ]:
# Check karte hain kya ye worst-predicted movies "popular" (highly-rated on average) hain
popularity_stats = train.groupby('item_id')['rating'].agg(['mean', 'count'])

worst_item_ids = worst_predictions['item_id'].tolist()
worst_popularity = popularity_stats.loc[worst_item_ids].merge(
    movies[['item_id', 'title']], left_index=True, right_on='item_id'
)[['title', 'mean', 'count']]

worst_popularity.columns = ['title', 'avg_rating_in_train', 'num_ratings_in_train']
print("Popularity stats for the worst-predicted movies:")
worst_popularity

Popularity stats for the worst-predicted movies:


,title,avg_rating_in_train,num_ratings_in_train
7,Babe (1995),3.936047,172
570,Another Stakeout (1993),2.458333,24
143,Die Hard (1988),3.853659,205
525,Ben-Hur (1959),3.890000,100
142,"Sound of Music, The (1965)",3.816092,174
287,Scream (1996),3.428205,390
698,Little Women (1994),3.736842,76
513,Annie Hall (1977),3.872483,149
50,Legends of the Fall (1994),3.453125,64
63,"Shawshank Redemption, The (1994)",4.456140,228


## Error Analysis Findings

### Pattern Identified
The worst predictions almost all follow the same pattern: the model predicts a **high rating
(4.5–5)** for movies that are, on average, highly-rated in the training data (e.g. Shawshank
Redemption, Ben-Hur, Annie Hall), but the specific test user rated them very **low (1)**.

### Root Cause
This is a classic **cold-start / sparse-data limitation** of collaborative filtering:
- For users with very few ratings in the training set, the model has limited signal to learn
  their individual taste, so it falls back toward predicting values closer to the item's
  overall popularity/average rating.
- The model correctly learned that these movies are *generally* well-liked, but failed to
  capture the *specific* user's minority opinion, since it didn't have enough interaction
  data for that user to override the general trend.

### Implication
This directly motivates the next phase of the project: a **hybrid model** combining
collaborative filtering with content-based filtering, which can use item metadata (genre,
etc.) to make better predictions even when a user's rating history is sparse — rather than
relying purely on the popularity signal learned by collaborative filtering.

### Example Failed Predictions
| User | Movie | Actual Rating | Predicted Rating | Error |
|---|---|---|---|---|
| 1 | Babe (1995) | 1 | 5.00 | 4.00 |
| 405 | Another Stakeout (1993) | 5 | 1.00 | 4.00 |
| 312 | Die Hard (1988) | 1 | 4.98 | 3.98 |
| 38 | Ben-Hur (1959) | 1 | 4.85 | 3.85 |